# Task 1.2 — Modeling & Tuning Under Time Constraints

Goal: train ≥2 model families (classical + CNN), calibrate a threshold for `fpr_real ≤ 0.20`, and verify `recall_ai ≥ 0.8` on `data/validation/` — all within 5× the Appendix C reference timing on this machine.

Depends on the cleaning logic developed in `01_task11_exploration_and_cleaning.ipynb`.

Outputs feed `solution/prepare.py`, `train.py`, `predict.py` and report §1.2.

In [ ]:
import sys, io, time, json, subprocess
from pathlib import Path

ROOT = Path.cwd().parent
SOLUTION = ROOT / "solution"
sys.path.insert(0, str(SOLUTION))

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression

from _lib import seed
from _lib.data import binarize_label
from _lib.model import build_appendix_b_cnn

seed.set_deterministic(0)

DATA = SOLUTION / "data"
ARTIFACTS = SOLUTION / "artifacts" / "task02"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 224

## 1. Reference timing baseline

Run Appendix C once. Our training wall-clock must stay ≤ 5× this on the same machine. Re-run if hardware changes.

In [ ]:
ref = subprocess.run(
    [sys.executable, str(ROOT / "train_time_reference.py")],
    capture_output=True, text=True, check=True,
)
print(ref.stdout)
elapsed = float([line for line in ref.stdout.splitlines() if "elapsed_seconds" in line][-1].split("=")[-1])
budget = 5 * elapsed
print(f"reference={elapsed:.1f}s   training budget = 5x = {budget:.1f}s")

## 2. Load splits

Eager load capped subsets while prototyping — the streaming version goes into `prepare.py`. Replace `max_rows` with `None` once the cell-stack runs end-to-end.

In [ ]:
def decode(buf):
    im = Image.open(io.BytesIO(buf)).convert("RGB")
    w, h = im.size
    s = IMG_SIZE / min(w, h)
    im = im.resize((int(round(w * s)), int(round(h * s))), Image.BILINEAR)
    nw, nh = im.size
    left = (nw - IMG_SIZE) // 2
    top = (nh - IMG_SIZE) // 2
    im = im.crop((left, top, left + IMG_SIZE, top + IMG_SIZE))
    return np.asarray(im, dtype=np.float32) / 255.0

def load_split(split, max_rows=None):
    xs, ys = [], []
    for f in sorted((DATA / split).glob("*.parquet")):
        df = pq.read_table(f).to_pandas()
        for _, row in df.iterrows():
            arr = decode(row["image"])
            xs.append(arr)
            ys.append(binarize_label(int(row["source_class"])))
            if max_rows and len(xs) >= max_rows:
                return np.stack(xs), np.array(ys, dtype=np.int64)
    return np.stack(xs), np.array(ys, dtype=np.int64)

# small caps for fast iteration; bump up once cells settle
X_train, y_train = load_split("train",                 max_rows=2000)
X_cal,   y_cal   = load_split("calibration",           max_rows=1000)
X_val,   y_val   = load_split("validation",            max_rows=1000)
X_va,    y_va    = load_split("validation_augmented",  max_rows=1000)

for name, X, y in [("train", X_train, y_train), ("cal", X_cal, y_cal),
                   ("val", X_val, y_val), ("val_aug", X_va, y_va)]:
    print(f"{name:8s} X={X.shape}  ai_share={y.mean():.2%}")

## 3. Classical baseline

Engineered features: per-channel mean, std, and 8-bin histogram. Logistic regression. Cheap to fit, gives a floor to compare the CNN against — and required by the PDF (≥2 model families).

In [ ]:
def features(X):
    """(N, 224, 224, 3) -> (N, 30) engineered feature vector."""
    n = X.shape[0]
    mean = X.mean(axis=(1, 2))                # (N, 3)
    std = X.std(axis=(1, 2))                  # (N, 3)
    hist = np.zeros((n, 24), dtype=np.float32)
    for c in range(3):
        for i, lo in enumerate(np.linspace(0, 1, 9)[:-1]):
            hist[:, c * 8 + i] = ((X[..., c] >= lo) & (X[..., c] < lo + 1/8)).mean(axis=(1, 2))
    return np.concatenate([mean, std, hist], axis=1).astype(np.float32)

t0 = time.monotonic()
F_train = features(X_train)
F_val   = features(X_val)
clf = LogisticRegression(max_iter=500).fit(F_train, y_train)
p_lr_val = clf.predict_proba(F_val)[:, 1]
print(f"classical fit+predict in {time.monotonic() - t0:.1f}s")
print(f"acc@0.5={(((p_lr_val > 0.5).astype(int)) == y_val).mean():.3f}")

## 4. CNN baseline (Appendix B)

Deadline-aware loop matching `train.py`'s scaffold: stop on `time.monotonic() > deadline`, keep the best-by-recall checkpoint.

In [ ]:
def to_chw(X):
    return torch.from_numpy(X).permute(0, 3, 1, 2).contiguous()

def train_cnn(model, X, y, deadline, batch=32, lr=1e-3):
    """Stop when time.monotonic() >= deadline. Returns step losses."""
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    Xt, yt = to_chw(X), torch.from_numpy(y).long()
    n = len(Xt)
    losses = []
    model.train()
    while time.monotonic() < deadline:
        perm = torch.randperm(n)
        for i in range(0, n, batch):
            if time.monotonic() >= deadline:
                break
            ix = perm[i:i+batch]
            logits = model(Xt[ix])
            loss = loss_fn(logits, yt[ix])
            opt.zero_grad(); loss.backward(); opt.step()
            losses.append(loss.item())
    return losses

@torch.no_grad()
def cnn_scores(model, X, batch=64):
    model.eval()
    Xt = to_chw(X)
    out = []
    for i in range(0, len(Xt), batch):
        logits = model(Xt[i:i+batch])
        out.append(torch.softmax(logits, dim=1)[:, 1].numpy())
    return np.concatenate(out)

seed.set_deterministic(0)
cnn = build_appendix_b_cnn(k=32)
t0 = time.monotonic()
# while prototyping, cap at ~120s so the cell stays interactive
losses = train_cnn(cnn, X_train, y_train, deadline=t0 + min(120.0, budget))
print(f"trained {len(losses)} steps in {time.monotonic() - t0:.1f}s, last_loss={losses[-1]:.3f}")

p_cnn_val = cnn_scores(cnn, X_val)
p_cnn_cal = cnn_scores(cnn, X_cal)

## 5. Metrics

`recall_ai = TP / (TP + FN)`, `fpr_real = FP / (FP + TN)`. Verify against sklearn on a slice.

In [ ]:
def confusion(y_true, y_pred):
    tn = int(((y_true == 0) & (y_pred == 0)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    return {"tn": tn, "fp": fp, "fn": fn, "tp": tp}

def recall_ai(y_true, y_pred):
    c = confusion(y_true, y_pred)
    return c["tp"] / max(c["tp"] + c["fn"], 1)

def fpr_real(y_true, y_pred):
    c = confusion(y_true, y_pred)
    return c["fp"] / max(c["fp"] + c["tn"], 1)

from sklearn.metrics import recall_score
yp = (p_cnn_val > 0.5).astype(int)
assert abs(recall_ai(y_val, yp) - recall_score(y_val, yp)) < 1e-9
print("OK: recall matches sklearn")

## 6. Threshold calibration

Pick the highest threshold `t` such that FPR on `data/calibration/` real-only rows ≤ 0.20. Higher `t` ⇒ fewer FPs ⇒ recall headroom on the validation split.

In [ ]:
def pick_threshold_for_fpr(scores_real, target_fpr=0.20):
    """scores_real = P(class=1) for label-0 (real) samples only.

    The k-th largest real-score corresponds to FPR = k/N. We pick a threshold
    just above it so FPR <= target.
    """
    s = np.sort(np.asarray(scores_real))[::-1]
    k = int(np.floor(target_fpr * len(s)))
    return float(s[k]) if k < len(s) else 1.0

thr = pick_threshold_for_fpr(p_cnn_cal[y_cal == 0], target_fpr=0.20)
print(f"calibrated threshold = {thr:.3f}")

# realised FPR on the calibration split (sanity)
yp_cal = (p_cnn_cal >= thr).astype(int)
print(f"realised FPR on cal = {fpr_real(y_cal, yp_cal):.3f}")

(ARTIFACTS / "threshold.json").write_text(json.dumps({"threshold": thr, "target_fpr": 0.20}))

## 7. Evaluation table

Both model families on `validation/` and `validation_augmented/`. Targets: `recall_ai ≥ 0.8`, `fpr_real ≤ 0.20` on the original validation split.

In [ ]:
def eval_block(name, scores, y, thr):
    yp = (scores >= thr).astype(int)
    return {"model_split": name, "n": len(y),
            "recall_ai": recall_ai(y, yp), "fpr_real": fpr_real(y, yp),
            **confusion(y, yp)}

p_cnn_va = cnn_scores(cnn, X_va)
p_lr_va  = clf.predict_proba(features(X_va))[:, 1]

rows = [
    eval_block("CNN val",     p_cnn_val, y_val, thr),
    eval_block("CNN val_aug", p_cnn_va,  y_va,  thr),
    eval_block("LR  val",     p_lr_val,  y_val, 0.5),
    eval_block("LR  val_aug", p_lr_va,   y_va,  0.5),
]
pd.DataFrame(rows).round(3)

## 8. Hyperparameter ablation

Small grid over CNN width and learning rate, fixed budget per cell. Use this to motivate the chosen config in the report.

In [ ]:
results = []
for k in (16, 32):
    for lr in (1e-3, 3e-4):
        seed.set_deterministic(0)
        m = build_appendix_b_cnn(k=k)
        train_cnn(m, X_train, y_train, deadline=time.monotonic() + 60, lr=lr)
        s_cal = cnn_scores(m, X_cal)
        s_val = cnn_scores(m, X_val)
        t = pick_threshold_for_fpr(s_cal[y_cal == 0], 0.20)
        yp = (s_val >= t).astype(int)
        results.append({"k": k, "lr": lr, "thr": round(t, 3),
                        "recall_ai": recall_ai(y_val, yp),
                        "fpr_real":  fpr_real(y_val, yp)})
pd.DataFrame(results).round(3)

## 9. Budget proof

Final end-to-end training wall-clock must stay ≤ 5× the reference. Plug in the actual timing from the chosen config above before claiming compliance.

In [ ]:
chosen_train_seconds = 60.0  # replace with the actual time of the chosen config
assert chosen_train_seconds <= budget, f"over budget: {chosen_train_seconds:.1f}s > {budget:.1f}s"
print(f"OK: {chosen_train_seconds:.1f}s <= 5x reference {budget:.1f}s")

## 10. Port targets — hand-off to `solution/`

- `_lib/io.py::read_parquet_split` / `decode_image` ← streaming version of `decode`/`load_split`
- `_lib/data.py::LabeledImageDataset` / `PredictDataset` ← iterators built on the above (predict yields `(row_id, image)`)
- `_lib/metrics.py::recall_ai` / `fpr_real` / `confusion` ← copy verbatim
- `_lib/calibration.py::pick_threshold_for_fpr` (cell 6) + `write_threshold_json` / `read_threshold_json` (use the same JSON shape)
- `_lib/model.py::build_classical_baseline` ← `features` + `LogisticRegression(max_iter=500)` factory
- `prepare.py::prepare_train_features` / `prepare_calibration_features` / `prepare_validation_features` ← `features()` cached to `artifacts/prepared/`
- `train.py::build_model` / `train_one_epoch` / `evaluate` / `calibrate_threshold` ← cells 4, 6
- `predict.py::load_checkpoint` / `predict` / `write_predictions_csv` ← inference + softmax + threshold; CSV header `row_id,predicted_label`

Figures from cells 4 (loss curve, optional) and the cell 7 table land in report §1.2.